In [5]:
import cv2
import torch
import torch.nn as nn
from torchvision.models import resnet18, ResNet18_Weights
import torchvision.transforms as T
import numpy as np

# ==========================================
# ۱. کلاس مدل شبکه عصبی
# ==========================================
class AccidentBiLSTM(nn.Module):
    def __init__(self, input_size=512, hidden_size=256, num_layers=1, dropout=0.0):
        super(AccidentBiLSTM, self).__init__()
        self.lstm = nn.LSTM(input_size, hidden_size, num_layers, batch_first=True, bidirectional=True)
        self.fc1 = nn.Linear(hidden_size * 2, 64)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(dropout)
        self.fc2 = nn.Linear(64, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        pooled_out, _ = torch.max(lstm_out, dim=1) 
        x = self.dropout(self.relu(self.fc1(pooled_out)))
        return self.sigmoid(self.fc2(x)).squeeze(-1)

# ==========================================
# ۲. لود کردن مدل‌ها
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print("در حال بارگذاری مدل‌ها...")

saved_weights = torch.load('best_final_bilstm.pth', map_location=device, weights_only=True)
detected_hidden = saved_weights['fc1.weight'].shape[1] // 2
detected_layers = sum(1 for k in saved_weights.keys() if k.startswith('lstm.weight_ih_l') and 'reverse' not in k)

bilstm_model = AccidentBiLSTM(input_size=512, hidden_size=detected_hidden, num_layers=detected_layers).to(device)
bilstm_model.load_state_dict(saved_weights)
bilstm_model.eval()

resnet = resnet18(weights=ResNet18_Weights.DEFAULT)
resnet = nn.Sequential(*list(resnet.children())[:-1]).to(device)
resnet.eval()

# ==========================================
# ۳. تابع کمکی برای متون
# ==========================================
def draw_soft_text(img, text, pos, scale=0.6, color=(255, 255, 255), thickness=1, align="left"):
    font = cv2.FONT_HERSHEY_SIMPLEX
    if align == "right":
        (tw, th), _ = cv2.getTextSize(text, font, scale, thickness)
        pos = (pos[0] - tw, pos[1])
    x, y = pos
    cv2.putText(img, text, (x + 1, y + 1), font, scale, (15, 15, 15), thickness + 1, cv2.LINE_AA)
    cv2.putText(img, text, (x, y), font, scale, color, thickness, cv2.LINE_AA)

# ==========================================
# ۴. پردازشگر اصلی (نسخه مینیمال و شیشه‌ای)
# ==========================================
def create_pro_demo(video_path, output_name="PRO_DEMO.mp4", threshold=0.50):
    print(f"\n📊 در حال پردازش نسخه مینیمال: {video_path}")
    cap = cv2.VideoCapture(video_path)
    
    raw_frames = []
    frames_tensor = []
    
    while True:
        ret, frame = cap.read()
        if not ret: break
        raw_frames.append(frame)
        rgb_frame = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
        frame_t = torch.from_numpy(rgb_frame).permute(2, 0, 1).float() / 255.0
        frame_t = T.Resize((224, 224), antialias=True)(frame_t)
        frames_tensor.append(frame_t)
    cap.release()
    
    if not frames_tensor: return
    
    with torch.no_grad():
        video_batch = torch.stack(frames_tensor).to(device)
        features = [resnet(video_batch[i:i+32]).squeeze(-1).squeeze(-1) for i in range(0, len(video_batch), 32)]
        video_features = torch.cat(features, dim=0)

    orig_H, orig_W = raw_frames[0].shape[:2]
    TARGET_W = 800  
    TARGET_H = int(orig_H * (TARGET_W / orig_W))
    CHART_H = 180 
    
    out = cv2.VideoWriter(output_name, cv2.VideoWriter_fourcc(*'mp4v'), 10, (TARGET_W, TARGET_H + CHART_H))
    
    trail_layer = np.zeros((TARGET_H, TARGET_W, 3), dtype=np.float32)
    ai_probabilities = []
    
    for i in range(1, len(raw_frames)):
        curr_orig = raw_frames[i]
        prev_orig = raw_frames[i-1]
        
        curr = cv2.resize(curr_orig, (TARGET_W, TARGET_H))
        prev = cv2.resize(prev_orig, (TARGET_W, TARGET_H))
        
        with torch.no_grad():
            current_feats = video_features[:i]
            seq_len = current_feats.size(0)
            if seq_len < 100:
                padding = torch.zeros(100 - seq_len, 512).to(device)
                current_feats = torch.cat((current_feats, padding), dim=0)
            else:
                current_feats = current_feats[:100, :]
                
            current_prob = bilstm_model(current_feats.unsqueeze(0)).item()
            ai_probabilities.append(current_prob)
            
        is_accident_now = current_prob >= threshold

        # --- پردازش حرکت (بسیار نرم و گرادیانی) ---
        diff = cv2.absdiff(curr, prev)
        gray_diff = cv2.cvtColor(diff, cv2.COLOR_BGR2GRAY)
        
        # تقویت ملایم تفاوت‌ها
        gray_diff = cv2.convertScaleAbs(gray_diff, alpha=2.5)
        # استفاده از TOZERO به جای BINARY برای حفظ گرادیان طبیعی لبه‌ها
        _, mask = cv2.threshold(gray_diff, 25, 255, cv2.THRESH_TOZERO)
        # بلورِ بسیار بزرگتر برای ایجاد حالت هاله‌ی گازی (Glow)
        mask = cv2.GaussianBlur(mask, (21, 21), 0)
        
        # ساخت لایه آلفا (شفافیت) از مسک برای جلوگیری از غلبه‌ی رنگ هیت‌مپ
        mask_3d = cv2.cvtColor(mask, cv2.COLOR_GRAY2BGR).astype(np.float32) / 255.0
        
        if is_accident_now:
            heatmap = cv2.applyColorMap(mask, cv2.COLORMAP_JET)
            # ضرب هیت‌مپ در آلفا: پس‌زمینه‌ی آبی کاملاً حذف و محو می‌شود
            color_mask = heatmap.astype(np.float32) * mask_3d
        else:
            color_mask = np.zeros_like(curr, dtype=np.float32)
            color_mask[:, :, 0] = mask.astype(np.float32)        # آبی
            color_mask[:, :, 1] = mask.astype(np.float32) * 0.7  # فیروزه‌ای ملایم
            color_mask[:, :, 2] = 0
            
        # محو شدن سریع‌تر ردپا (0.70) و اضافه شدنِ بسیار نرم‌ترِ رنگ جدید (0.40)
        trail_layer = trail_layer * 0.80 + color_mask * 0.70
        trail_layer_clipped = np.clip(trail_layer, 0, 255).astype(np.uint8)
        
        # ترکیب نهایی: ویدیوی اصلی 85% وضوح دارد، افکت فقط 55% روی آن می‌نشیند
        main_video = cv2.addWeighted(curr, 0.85, trail_layer_clipped, 0.80, 0)
        
        # --- رسم پنل نمودار تحلیلی (بدون تغییر) ---
        chart_panel = np.zeros((CHART_H, TARGET_W, 3), dtype=np.uint8)
        chart_panel[:] = (20, 20, 24) 
        
        for grid_p in [0.25, 0.50, 0.75]:
            gy = int(CHART_H - (grid_p * CHART_H))
            cv2.line(chart_panel, (0, gy), (TARGET_W, gy), (40, 40, 45), 1)
        
        thresh_y = int(CHART_H - (threshold * CHART_H))
        cv2.line(chart_panel, (0, thresh_y), (TARGET_W, thresh_y), (80, 80, 80), 1, cv2.LINE_AA)
        
        draw_soft_text(chart_panel, f"Threshold {int(threshold*100)}%", 
                       (TARGET_W - 20, thresh_y - 8), scale=0.45, color=(160, 160, 160), align="right")
        
        points = []
        total_frames = len(raw_frames)
        for idx, p in enumerate(ai_probabilities):
            x = int((idx / total_frames) * TARGET_W)
            y = int(CHART_H - (p * CHART_H))
            points.append((x, y))
            
        if len(points) > 1:
            for j in range(1, len(points)):
                pt1 = points[j-1]
                pt2 = points[j]
                
                color = (255, 200, 0) if ai_probabilities[j] < threshold else (50, 50, 255)
                
                pts = np.array([[pt1[0], CHART_H], pt1, pt2, [pt2[0], CHART_H]], np.int32)
                overlay = chart_panel.copy()
                cv2.fillPoly(overlay, [pts], color)
                cv2.addWeighted(overlay, 0.15, chart_panel, 0.85, 0, chart_panel) 
                
                cv2.line(chart_panel, pt1, pt2, color, 2, cv2.LINE_AA)
                
            current_pt = points[-1]
            cv2.circle(chart_panel, current_pt, 5, (255, 255, 255), -1, cv2.LINE_AA)
            
        risk_color = (50, 50, 255) if is_accident_now else (255, 255, 255)
        draw_soft_text(chart_panel, f"AI RISK: {current_prob*100:.1f}%", 
                       (20, 35), scale=0.7, color=risk_color, thickness=2, align="left")
        
        status_text = "[ CRASH DETECTED ]" if is_accident_now else "[ MONITORING ]"
        status_color = (50, 50, 255) if is_accident_now else (255, 200, 0)
        draw_soft_text(chart_panel, status_text, 
                       (TARGET_W - 20, 35), scale=0.7, color=status_color, thickness=2, align="right")
        
        final_frame = np.vstack((main_video, chart_panel))
        
        times_to_write = 4 if is_accident_now else 1
        for _ in range(times_to_write):
            out.write(final_frame)

    out.release()
    print(f"✅ ویدیوی نهایی ساخته شد: {output_name}")

# اجرا
create_pro_demo('./accident_demo_1.mp4', "FINAL_SOFT_CRASH.mp4")
create_pro_demo('./normal_demo_1.mp4', "FINAL_SOFT_NORMAL.mp4")

در حال بارگذاری مدل‌ها...

📊 در حال پردازش نسخه مینیمال: ./accident_demo_1.mp4
✅ ویدیوی نهایی ساخته شد: FINAL_SOFT_CRASH.mp4

📊 در حال پردازش نسخه مینیمال: ./normal_demo_1.mp4
✅ ویدیوی نهایی ساخته شد: FINAL_SOFT_NORMAL.mp4



### هدف

این سلول یک ویدئوی دمو می‌سازد که روی آن دو لایه اضافه می‌شود.

اول، هیت‌مپ نرم روی فریم‌های ویدئو.

دوم، یک پنل نمودار در پایین صفحه که احتمال تصادف را لحظه به لحظه نشان می‌دهد.

هدف این است که نتیجه‌ی مدل به شکل بصری و قابل فهم برای بیننده نمایش داده شود.

### بخش ۱: مدل شبکه عصبی

کلاس

`AccidentBiLSTM`

این کلاس همان مدل Try #4 است که در نوت‌بوک آموزش دید.

دوطرفه

`bidirectional=True`

یک لایه LSTM دوطرفه.

یک لایه Linear با ۶۴ نورون.

تابع فعال‌سازی

`ReLU`

یک لایه Linear نهایی با یک خروجی.

تابع

`Sigmoid`

نکته‌ی مهم: اینجا از ماسک کردن پدینگ استفاده نکردیم. چون ورودی این سلول دقیقاً ۱۰۰ فریم (پد شده) است و همان‌طور که مدل آموزش دیده، پردازش می‌شود.

### بخش ۲: لود کردن مدل‌ها

دو مدل لود می‌شود.

اول، مدل

`BiLSTM`

از فایل

`best_final_bilstm.pth`

 ابعاد مدل را از خود وزن‌ها استخراج می‌کنیم.

`detected_hidden`

تعداد نورون‌های پنهان را از شکل وزن لایه اول حساب می‌کند.

`detected_layers`

تعداد لایه‌های LSTM را از اسم پارامترها حساب می‌کند.

مزیت این کار: اگر مدل را با ابرپارامترهای مختلف آموزش داده باشیم، این کد خودکار تشخیص می‌دهد.

دوم،

`ResNet18`

برای استخراج ویژگی از فریم‌های ویدئو.

لایه آخرش حذف شده تا فقط بردار ۵۱۲ بعدی برگردد.

### بخش ۳: تابع

`draw_soft_text`

این تابع متن را روی تصویر می‌نویسد.

نکته‌ی خاص: به جای یک لایه متن، دو لایه می‌نویسد.

اول، یک لایه تیره در پس‌زمینه با فاصله یک پیکسل.

دوم، لایه اصلی متن.

نتیجه: متن با یک سایه‌ی ملایم نمایش داده می‌شود و خوانایی روی هر پس‌زمینه‌ای خوب است.

### بخش ۴: تابع اصلی

`create_pro_demo`

این تابع سه کار انجام می‌دهد.

اول، خواندن ویدئو و استخراج ویژگی.

دوم، پیش‌بینی احتمال تصادف برای هر فریم.

سوم، ساخت ویدئوی خروجی با افکت بصری.

### مرحله اول: خواندن فریم‌ها

تمام فریم‌های ویدئو خوانده می‌شوند.

هر فریم به ابعاد

`۲۲۴ در ۲۲۴`

تغییر اندازه می‌دهد و به تنسور تبدیل می‌شود.

نکته: تمام فریم‌ها یکجا نگه داشته می‌شوند نه پخش زنده. چون این دمو برای تحلیل آفلاین است.

### مرحله دوم: استخراج ویژگی و پیش‌بینی

فریم‌ها به دسته‌های ۳۲ تایی تقسیم می‌شوند.

هر دسته از

`ResNet18`

عبور می‌کند.

خروجی: یک بردار ۵۱۲ بعدی برای هر فریم.

سپس برای هر فریم این کار انجام می‌شود.

فریم‌های ۰ تا i (یعنی فقط گذشته) به مدل داده می‌شود.

مدل احتمال تصادف را برمی‌گرداند.

نکته‌ی مهم: از فریم‌های آینده استفاده نمی‌کنیم. یعنی پیش‌بینی

causal

است. این همان الگویی است که در سیستم Real-Time استفاده می‌شود.

### مرحله سوم: افکت هیت‌مپ نرم

اینجا نکته‌ی اصلی این سلول است.

به جای رسم هیت‌مپ معمولی که همه‌ی صحنه را رنگی می‌کند، از یک روش نرم استفاده می‌شود.

گام اول: اختلاف بین دو فریم متوالی محاسبه می‌شود.

`cv2.absdiff`

این اختلاف نشان می‌دهد چه چیزی در صحنه حرکت کرده.

گام دوم: اختلاف تقویت می‌شود.

مقدار تقویت

`alpha=2.5`

گام سوم: با استفاده از

`cv2.THRESH_TOZERO`

مقادیر کم حذف می‌شوند ولی گرادیان طبیعی حفظ می‌شود.

برخلاف

`BINARY`

که فقط صفر و ۲۵۵ دارد، این روش طیف پیوسته‌ای از شدت‌ها را نگه می‌دارد.

گام چهارم: یک بلور بزرگ با اندازه

`۲۱ در ۲۱`

اعمال می‌شود.

نتیجه: افکت شبیه هاله‌ی گازی (Glow) می‌شود، نه لکه‌های تیز.

### نکته‌ی ظریف: ماسک آلفا

یک نکته‌ی مهم در این روش وجود دارد.

اگر هیت‌مپ را مستقیم روی تصویر بگذاریم، پس‌زمینه‌ی آبی رنگ

Jet

همه‌جا دیده می‌شود حتی جاهایی که حرکتی نیست.

راه‌حل: ماسک حرکت را به عنوان کانال آلفا استفاده می‌کنیم.

`mask_3d`

نتیجه: هیت‌مپ فقط روی نواحی حرکت دیده می‌شود و پس‌زمینه‌ی آبی کاملاً محو است.

### حالت‌های بصری

دو حالت داریم.

اگر احتمال تصادف پایین باشد

یعنی زیر آستانه.

رنگ افکت: آبی و فیروزه‌ای ملایم.

اگر احتمال تصادف بالا باشد

یعنی بالای آستانه.

رنگ افکت: هیت‌مپ

Jet

کامل.

### ردپای حرکتی

یک لایه‌ی تجمعی به نام

`trail_layer`

نگه داشته می‌شود که اثر حرکات گذشته را حفظ می‌کند.

هر فریم:

مقدار قبلی ضرب در

`0.80`

می‌شود، یعنی کمی محو می‌شود.

مقدار جدید ضرب در

`0.70`

اضافه می‌شود.

نتیجه: یک ردپای نرم که با تأخیر محو می‌شود.

### مرحله چهارم: پنل نمودار تحلیلی

پایین هر فریم، یک پنل

`۱۸۰ پیکسل`

اضافه می‌شود.

این پنل سه چیز دارد.

اول، گرید افقی در سه سطح.

دوم، خط آستانه که با مقدار

`threshold`

هم‌تراز است.

سوم، نمودار احتمال در طول زمان.

نمودار به شکل خطی است که زیرش پر می‌شود.

رنگ خط:

آبی کهربایی برای مقادیر زیر آستانه.

قرمز برای مقادیر بالای آستانه.

نقطه‌ی پایانی (فریم فعلی) با یک دایره‌ی سفید مشخص می‌شود.

### متن‌های روی پنل

دو متن نمایش داده می‌شود.

سمت چپ:

`AI RISK: XX.X%`

یعنی درصد احتمال تصادف در این فریم.

سمت راست:

`[ MONITORING ]` یا `[ CRASH DETECTED ]`